# 1 · Preprocess a light curve

The encoder takes light curves that have been through the
same preparation the training set went through, and it is conditioned on metadata that
describes the scale that preparation removed.

This notebook runs that preparation end to end, with `encotess.preprocess`:

1. **download** a SPOC 2-min light curve under an explicit TESS quality bitmask,
2. **clean** it — drop non-finite samples, rebase the time axis,
3. **normalize** the flux robustly, on the median and the 16–84 semi-interpercentile range,
4. **assemble** the 13 raw metadata fields,
5. **standardize** them into the values the network actually sees.

### What this notebook does *not* do

The stellar parameters are taken as given. Crossmatching a TIC to Gaia DR3, correcting
the parallax zero point, and dereddening the photometry to `G0` / `BPRP0` all happen
**upstream of EncoTESS**. Step 4 below documents the exact
conventions the released values follow, so whatever pipeline you use, match them.

> Needs network access and the preprocessing extras:
> `pip install -e ".[preprocess]"`

In [ ]:
import numpy as np
import encotess
from encotess import preprocess as pp

print('metadata fields:', encotess.DEFAULT_METADATA_FIELDS)

## The target

We use a star that is already in the released dataset, so every number we compute can be
checked against the shipped metadata rather than taken on trust: **TIC 344097064,
sector 15** — a thick-disk star.

The bundled CSVs give us both the per-light-curve row (the flux statistics we are about
to recompute) and the per-star row (the stellar parameters we would otherwise have to
source ourselves).

In [ ]:
import csv
from encotess import assets

TIC, SECTOR = 344097064, 15

def _find(path, **match):
    with open(path, newline='') as f:
        for row in csv.DictReader(f):
            if all(row[k] == str(v) for k, v in match.items()):
                return row
    raise LookupError(f'no row matching {match} in {path.name}')

sector_row = _find(assets.metadata_path('sector'), TIC_ID=TIC, sector=SECTOR)
star_row   = _find(assets.metadata_path('thickdisk_star'),
                   GaiaDR3_ID=sector_row['GaiaDR3_ID'])

print('Gaia DR3', sector_row['GaiaDR3_ID'], '| Tmag', sector_row['Tmag'])
print('released median_flux  ', sector_row['median_flux'])
print('released iqr_half_flux', sector_row['iqr_half_flux'])
print('released camera/ccd   ', sector_row['camera'], sector_row['ccd'])

## 1 · Download

`download_lightcurve` wraps the lightkurve search the training set was built with:
SPOC pipeline, 120 s exposures, PDCSAP flux, filtered to the requested TIC so a crowded
cone search does not hand back a neighbour.

### The quality bitmask

TESS flags bad cadences in a `QUALITY` column, and the bitmask decides which flags cause
a cadence to be dropped. **Pass the integer, not the string `'default'`** — lightkurve's
`'default'` resolved to `175` in versions ≤ 2.5.0 and to `17087` from 2.5.1 onward, so a
string makes your result depend on which lightkurve you happen to have installed.
`encotess.DEFAULT_QUALITY_BITMASK` is `17087`.

> **Reproducing the released banks exactly.** The banks were not all built under one
> mask. The thick-disk bank (this star) used 17087; the pretrain and hosts banks were
> built with lightkurve's then-default 175, which keeps the flag-512 "impulsive outlier"
> cadences that 17087 drops — about 0.016% of stored cadences. For a pretrain or hosts
> star, pass `quality_bitmask=175` if you need to match the released encoding
> cadence-for-cadence.

In [ ]:
raw = pp.download_lightcurve(TIC, SECTOR, quality_bitmask=17087)

print(f"cadences downloaded : {len(raw['flux'])}")
print(f"camera / ccd        : {raw['camera']} / {raw['ccd']}   (released: "
      f"{float(sector_row['camera']):.0f} / {float(sector_row['ccd']):.0f})")
print(f"time span           : {raw['time'][-1] - raw['time'][0]:.2f} d (BTJD)")
print(f"flux                : median {np.nanmedian(raw['flux']):.1f} e-/s, "
      f"{np.isnan(raw['flux']).sum()} NaN")

## 2 · Clean

`clean_lightcurve` keeps cadences with **finite flux and finite time**, then rebases the
time axis to `t - t[0]` so it starts at zero days — the released H5 banks store it that
way, and the encoder's time-aware pooling reads elapsed time, not absolute BTJD.

Non-finite `flux_err` values are deliberately *kept* at this stage. An isolated bad error
bar should not cost you a cadence of flux; those get filled in the next step instead.

In [ ]:
flux, flux_err, time = pp.clean_lightcurve(raw['flux'], raw['flux_err'], raw['time'])

print(f"kept {len(flux)} of {len(raw['flux'])} cadences "
      f"({len(raw['flux']) - len(flux)} dropped as non-finite)")
print(f"time: {time[0]:.3f} -> {time[-1]:.3f} d")
print(f"flux_err still non-finite: {(~np.isfinite(flux_err)).sum()}")

## 3 · Normalize the flux

The encoder sees flux centred on its median and scaled by the **16–84 semi-interpercentile
range** — half the distance between the 16th and 84th percentiles:

$$
\mathrm{median\_flux} = \mathrm{median}(F), \qquad
\mathrm{iqr\_half\_flux} = \frac{P_{84}(F) - P_{16}(F)}{2}
$$

$$
F' = \frac{F - \mathrm{median\_flux}}{\mathrm{iqr\_half\_flux}}, \qquad
\sigma_F' = \frac{\sigma_F}{\mathrm{iqr\_half\_flux}}
$$

Not mean and standard deviation. Both of those are dragged around by a single flare or
one uncorrected systematic, and a light curve rescaled by an inflated `std` arrives at
the encoder with its real variability squashed. A Gaussian puts ±1σ at the 15.87th and
84.13th percentiles, so this half-range *is* σ on a well-behaved curve — the same scale,
estimated in a way that outliers cannot drag, which for a flare star is the interesting
case.

> **The field name `iqr_half_flux` is a misnomer**, kept for compatibility with the
> released CSVs and HDF5 banks. The quantity is the **semi-interpercentile range**, taken
> at the 16th and 84th percentiles — not half the interquartile range. Half the true IQR,
> `(P75 − P25) / 2`, would give you roughly 0.67σ and mis-scale the curve.

Both statistics are then handed to the encoder as the `median_flux` and `iqr_half_flux`
metadata fields, so the network still knows the absolute brightness and variability
amplitude that normalization just removed.

Normalized `flux_err` values that are non-finite are filled with that curve's median —
the encoder has no missing-value channel for `flux_err`.

In [ ]:
norm = pp.robust_normalize(flux, flux_err)

print(f"median_flux   : {norm['median_flux']:.6f}   released {float(sector_row['median_flux']):.6f}")
print(f"iqr_half_flux : {norm['iqr_half_flux']:.6f}   released {float(sector_row['iqr_half_flux']):.6f}")
for k in ('median_flux', 'iqr_half_flux'):
    rel = abs(norm[k] - float(sector_row[k])) / float(sector_row[k])
    print(f"  {k:14s} relative difference {rel:.2e}")

Expect agreement at the `1e-5` level or better, not bit-for-bit. The released statistics
were computed after a float32 round-trip through an earlier normalization step, so the
last digits differ; anything much larger than `1e-4` means something real has changed —
a different bitmask, a different SPOC data release, or the wrong sector.

How much does the robust choice buy you? On this star, almost nothing — it is a quiet
thick-disk dwarf, and `std` and the semi-interpercentile range agree to well under a
percent.
That *is* the argument: robust scaling costs you nothing on a clean curve and protects
you on a flaring one, where `std` can run several times larger and squash the real
variability before the encoder ever sees it.

In [ ]:
print(f"robust  : median {norm['median_flux']:10.3f}   semi-IPR {norm['iqr_half_flux']:8.3f}")
print(f"mean/std: mean   {flux.mean():10.3f}   std      {flux.std():8.3f}")
print(f"  -> std / semi-IPR = {flux.std() / norm['iqr_half_flux']:.3f}"
      "   (>1 means outliers inflated the scale)")
print()
print('normalized flux :', norm['flux'][:5].round(3), '...')
print('normalized err  :', norm['flux_err'][:5].round(3), '...')

## 4 · Assemble the metadata

`build_metadata` collects the 13 raw fields. Three sources:

| Source | Fields |
|---|---|
| Light-curve header | `sector`, `camera`, `ccd`, `cadence_s` |
| Step 3 | `median_flux`, `iqr_half_flux` |
| **You** | `Tmag`, `parallax`, `parallax_error`, `G0`, `G0_err`, `BPRP0`, `BPRP0_err` |

The conventions the released metadata follows — match them:

- **`Tmag`** — TESS magnitude from the TIC.
- **`parallax`** — Gaia DR3 parallax in **mas**, **zero-point corrected** (Lindegren
  et al. 2021; the `gaiadr3-zeropoint` package, applied where
  `astrometric_params_solved ∈ {31, 95}`). `parallax_error` is the catalogue error, mas.
- **`G0`, `BPRP0`** — **apparent** Gaia G magnitude and BP−RP colour, **dereddened**.
  `G0` is apparent, not absolute. `G0_err` / `BPRP0_err` are the propagated errors.

Any field you do not have may be omitted or passed as `NaN`: the encoder was trained with
a validity-mask channel and handles missing values. That is a graceful degradation, not a
free pass — a star missing its photometry is a star the encoder knows less about.

In [ ]:
metadata = pp.build_metadata(
    sector=raw['sector'], camera=raw['camera'], ccd=raw['ccd'],
    median_flux=norm['median_flux'], iqr_half_flux=norm['iqr_half_flux'],
    cadence_s=raw['cadence_s'],
    # Stellar parameters: already zero-point corrected and dereddened, upstream.
    Tmag=float(sector_row['Tmag']),
    parallax=float(star_row['parallax']),
    parallax_error=float(star_row['parallax_error']),
    G0=float(star_row['G0']),
    G0_err=float(star_row['G0_err']),
    BPRP0=float(star_row['BPRP0']),
    BPRP0_err=float(star_row['BPRP0_err']),
)

for field, value in metadata.items():
    print(f'  {field:16s} {value:14.5f}')

## 5 · Standardize

The network does not see those raw values. `MetadataStandardizer` applies fixed,
data-independent rules — no fitted statistics, so they are identical at training time and
on your star:

- **log10**: `cadence_s`, `Tmag`, `parallax`, `parallax_error`, `G0`, `G0_err`,
  `BPRP0_err`, `median_flux`, `iqr_half_flux`
- **÷ 100**: `sector`
- **raw**: `BPRP0`, `camera`, `ccd`

`encode()` applies this internally — you never have to call it. `preprocess.standardize`
exposes it so you can see what each field becomes and, more usefully, **which fields got
dropped**. A log10 field that is zero or negative has no representable value and is
flagged missing exactly like a `NaN`: a negative zero-point-corrected parallax silently
leaves the encoder with no distance information at all. Check the mask.

In [ ]:
features, mask = pp.standardize(metadata)

print(f"  {'field':16s} {'raw':>14s} {'standardized':>14s}   mask")
for field, rawv, stdv, m in zip(encotess.DEFAULT_METADATA_FIELDS,
                                metadata.values(), features, mask):
    flag = '' if m else '   <- MISSING'
    print(f'  {field:16s} {rawv:14.5f} {stdv:14.5f}   {m:.0f}{flag}')

missing = [f for f, m in zip(encotess.DEFAULT_METADATA_FIELDS, mask) if m == 0]
print('\nmissing fields:', missing or 'none')

## All of it in one call

`preprocess_lightcurve` runs steps 1–4 and hands back arrays `encode()` accepts directly.

In [ ]:
prepped = pp.preprocess_lightcurve(
    TIC, SECTOR,
    stellar_params={
        'Tmag': float(sector_row['Tmag']),
        'parallax': float(star_row['parallax']),
        'parallax_error': float(star_row['parallax_error']),
        'G0': float(star_row['G0']),
        'G0_err': float(star_row['G0_err']),
        'BPRP0': float(star_row['BPRP0']),
        'BPRP0_err': float(star_row['BPRP0_err']),
    },
)

enc = encotess.load_encoder(device='cpu')
z = enc.encode(prepped['flux'], prepped['flux_err'], prepped['time'],
               metadata=prepped['metadata'])

print('cadences   :', len(prepped['flux']))
print('latent     :', z.shape)
print('pca-16     :', enc.project_pca(z, dim=16)[:4].round(3), '...')

One more step happens inside `encode()`, not here: it trims **10 samples from each end**
of the curve (`trim_edges=10`, matching training) to drop TESS thermal-settling and
scattered-light artifacts. The flux statistics in step 3 are computed *before* that trim,
as they were in training, which is why `len(prepped['flux'])` is 20 longer than what the
encoder ingests. Leave `trim_edges` alone unless you know why you are changing it.

### Next

- **[2 · Encode a light curve](02_encode_a_lightcurve.ipynb)** — what the encoder does
  with these arrays, and the global PCA basis the package ships in.
- **[3 · Explore the released encodings](03_explore_the_encodings.ipynb)** — find this
  star's neighbours among every released light curve.